In [1]:
!pip install -q pykan

from google.colab import drive
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import gc

# Monta Google Drive
drive.mount('/content/drive')

# Caricamento del file CSV da Drive
file_path = '/content/drive/MyDrive/Tesi_CICIoT/train.csv'

print("Caricamento del dataset in corso...")
df = pd.read_csv(file_path)

# Creazione etichetta binaria
label_col = 'label' if 'label' in df.columns else 'Label'
is_benign = df[label_col].str.lower().str.contains('benign')
df['binary_label'] = (~is_benign).astype(int)

print(f"Totale righe caricate: {len(df)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 3.9 MB/s eta 0:00:00
Mounted at /content/drive
Caricamento del dataset in corso...
Totale righe caricate: 5491971


In [2]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import gc

print("Avvio campionamento stratificato per preservare gli attacchi rari...")

# 1. Isola i benigni
df_benign = df[df['binary_label'] == 0].sample(n=50000, random_state=42)

# 2. Isola gli attacchi
df_attack_full = df[df['binary_label'] == 1]
n_attack_samples = 50000

# 3. Campionamento stratificato proporzionale (senza groupby.apply per evitare deprecazioni)
df_attack = pd.concat([
    group.sample(n=max(1, int(len(group) * (n_attack_samples / len(df_attack_full)))), random_state=42)
    for _, group in df_attack_full.groupby(label_col)
]).reset_index(drop=True)

df_balanced = pd.concat([df_benign, df_attack]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Righe finali bilanciate: {len(df_balanced)}")

# Svuotiamo la RAM dal dataset originale
del df
del df_attack_full
gc.collect()
print("RAM liberata dai dataset intermedi.")

# Isolamento delle feature numeriche
feature_cols = df_balanced.select_dtypes(include=['float64', 'int64', 'float32', 'int32']).columns.tolist()
feature_cols = [c for c in feature_cols if c not in ['label', 'binary_label']]
print(f"Feature numeriche selezionate: {len(feature_cols)}")

# Estrazione feature e target
X_raw = df_balanced[feature_cols].values
y = df_balanced['binary_label'].values

# 1. Split Stratificato SUI DATI GREZZI (Prima della normalizzazione)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Fit e Transform SOLO SUL TRAIN SET
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)

# 3. Solo Transform SUL TEST SET (Nessun fit!)
X_test = scaler.transform(X_test_raw)

# Costruzione dei Tensori PyTorch per la GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDispositivo di addestramento attivo: {device}")

dataset = {
    'train_input': torch.FloatTensor(X_train).to(device),
    'train_label': torch.FloatTensor(y_train).reshape(-1, 1).to(device),
    'test_input': torch.FloatTensor(X_test).to(device),
    'test_label': torch.FloatTensor(y_test).reshape(-1, 1).to(device)
}

print(f"\n--- DATASET PRONTO PER KAN ---")
print(f"Train Input Shape: {dataset['train_input'].shape}")
print(f"Test Input Shape: {dataset['test_input'].shape}")
print(f"Bilanciamento Test Set (Media etichette): {y_test.mean():.2f} (Atteso 0.50)")

Avvio campionamento stratificato per preservare gli attacchi rari...
Righe finali bilanciate: 99984
RAM liberata dai dataset intermedi.
Feature numeriche selezionate: 46

Dispositivo di addestramento attivo: cuda

--- DATASET PRONTO PER KAN ---
Train Input Shape: torch.Size([79987, 46])
Test Input Shape: torch.Size([19997, 46])
Bilanciamento Test Set (Media etichette): 0.50 (Atteso 0.50)


In [3]:
from kan import KAN
from sklearn.metrics import precision_recall_fscore_support

print("Inizializzazione del modello KAN su GPU...")
input_dim = dataset['train_input'].shape[1]

# Architettura
model = KAN(width=[input_dim, 32, 16, 1], grid=5, k=3, seed=42).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Calcolo dinamico del pos_weight per la BCE Loss
num_neg = (y_train == 0).sum()
num_pos = (y_train == 1).sum()
pos_weight_val = num_neg / num_pos
pos_weight = torch.tensor([pos_weight_val], dtype=torch.float32).to(device)

print(f"Bilanciamento dinamico loss applicato (pos_weight): {pos_weight_val:.4f}")

# Funzione di costo pesata
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
epochs = 50
print(f"Avvio training per {epochs} epoche...")

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    train_out = model(dataset['train_input'])
    loss = criterion(train_out, dataset['train_label'])

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}] | Loss: {loss.item():.4f}')

print("\n--- VALUTAZIONE FINALE SUL CICIoT2023 ---")
model.eval()
with torch.no_grad():
    test_outputs = model(dataset['test_input'])
    preds = (torch.sigmoid(test_outputs) > 0.5).float().cpu().numpy()
    labels = dataset['test_label'].cpu().numpy()

    # Calcolo pulito e unico delle metriche
    accuracy = (preds == labels).mean()
    precision, recall, f1_binary, _ = precision_recall_fscore_support(labels, preds, average='binary')
    _, _, f1_macro, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)

    print(f"Accuracy:          {accuracy:.4f}")
    print(f"Precision:         {precision:.4f}")
    print(f"Recall:            {recall:.4f}")
    print(f"F1-Score (Binary): {f1_binary:.4f}")
    print(f"F1-Score (Macro):  {f1_macro:.4f}")

Inizializzazione del modello KAN su GPU...
checkpoint directory created: ./model
saving model version 0.0
Bilanciamento dinamico loss applicato (pos_weight): 1.0003
Avvio training per 50 epoche...
Epoch [10/50] | Loss: 0.6691
Epoch [20/50] | Loss: 0.6333
Epoch [30/50] | Loss: 0.5743
Epoch [40/50] | Loss: 0.4858
Epoch [50/50] | Loss: 0.3758

--- VALUTAZIONE FINALE SUL CICIoT2023 ---
Accuracy:          0.9395
Precision:         0.9156
Recall:            0.9682
F1-Score (Binary): 0.9412
F1-Score (Macro):  0.9394
